# Make predictions using models

Overall Goal
- Run trained Chemprop GNN ensembles in inference mode on large external compound libraries (Broad 37K, Broad ~800K, Enamine ~5M "easy-to-order") to score every molecule for predicted activity against *N. gonorrhoeae*.
- Drive each successive round of experimental prioritization: PK-only model on the Broad 37K HTS, PK+37K model on the 800K and 5M libraries, and the retrained PK+37K+first-round-validation model on the same 800K and 5M libraries.
- Produce per-compound prediction CSVs that the downstream filter notebook ([4A_5A_filter_predictions_to_prioritize_compounds_for_validation.ipynb](4A_5A_filter_predictions_to_prioritize_compounds_for_validation.ipynb)) consumes to assemble validation tiers and figure panels.

Relation to Manuscript
- Related to Figure 4A, Figure 5A.

Takeaways
- Predictions are run in chunks via `chemprop_predict` (subprocess) with ensemble variance enabled, then concatenated back into a single CSV per (model, library) pair.
- Two successive model retrains (PK → PK+37K → PK+37K+1st-round val) progressively shape which compounds get nominated, and outputs from this notebook are the direct inputs to the filtering/clustering step.

Helper Functions
- `chunk_up_and_run_predictions`: splits a CSV + matching `.npz` features file into chunks, calls `chemprop_predict` per chunk under the `chemprop` conda env (with `--ensemble_variance --no_features_scaling --gpu 0`), and recombines the per-chunk prediction CSVs into one DataFrame.

Scientific Methodology
- For each (library, model) pair, loaded the cleaned SMILES CSV and precomputed RDKit feature `.npz` from `../data/library_info/`, chunked them into 1000-molecule batches under `../out/predictions_from_models/chunks/`, and invoked `chemprop_predict`.
- Round 0 — PK-only model `FINAL151` on the Broad 37K HTS; output `37K_chunks_with_151_model.csv` (pre-screen prioritization).
- Round 1 — PK+37K model `FINALbayHO04052022` on Broad 800K (`broad800K_melis_predictions_04_25_2022.csv`) and on the 5M easy-to-order set (`extended_screen_set_melis_predictions_05_01_2022.csv`, drives Figure 4A).
- Round 2 — PK+37K+1st-round-val model `FINALbayHO11152022` on Broad 800K (`broad800K_melis_predictions_with_FINALbayHO11152022_11_16_2022.csv`, drives Figure 5A) and on the 5M easy-to-order set (`extended_screen_set_with_FINALbayHO11152022_melis_predictions_11_22_2022.csv`).

## Setup

In [ ]:
import subprocess

import numpy as np
import pandas as pd

%matplotlib inline

/Users/jackie16201/Desktop/Spring_2023/ngonorrhoeae_abx_ml_discovery/src/utils.py:5: DeprecationWarning: The rdkit.Chem.MCS module is deprecated; please use rdkit.Chem.rdFMCS instead.
  from rdkit.Chem import MCS, Descriptors, PandasTools, AllChem
<frozen importlib._bootstrap>:219: RuntimeWarning: to-Python converter for boost::shared_ptr<RDKit::FilterCatalogEntry const> already registered; second conversion method ignored.


## Helper functions

In [ ]:
def chunk_up_and_run_predictions(data_path, data_file, features_file, model_dir, chunksize=1000, smiles_col="SMILES"):
    """
    Splits a dataset into chunks, runs Chemprop predictions on each chunk, and recombines results.

    This function:
    1. Loads a CSV of input data and a NumPy features file.
    2. Splits both into chunks of size `chunksize` and saves temporary .csv and .npy files.
    3. Calls `chemprop_predict` via subprocess for each chunk, using the provided model directory.
    4. Collects and concatenates all prediction results into a single DataFrame.

    Parameters:
    data_path (str): Path to the directory containing the data and feature files.
    data_file (str): CSV file containing SMILES and other input data.
    features_file (str): NumPy .npz file containing precomputed features ('features' key required).
    model_dir (str): Path to a directory containing Chemprop model checkpoints.
    chunksize (int, optional): Number of molecules per chunk. Default is 1000.
    smiles_col (str, optional): Column name containing SMILES strings. Default is "SMILES".

    Returns:
    pd.DataFrame: Combined DataFrame of predictions from all chunks.

    Side Effects:
    - Writes chunked .csv and .npy files into `../out/predictions_from_models/chunks/`.
    - Executes shell commands with `chemprop_predict` for each chunk.
    - Saves prediction CSVs (`*_preds.csv`) in the same chunks directory.
    """
    # first chunk up data + features
    data = pd.read_csv(data_path + data_file)
    with np.load(data_path + features_file) as ftsdata:
        ftsdata = ftsdata["features"]

    temp_dir_chunks = "../out/predictions_from_models/chunks/"
    for i in range(0, len(data), chunksize):
        smis = pd.DataFrame(data.iloc[i : i + chunksize, :])  # noqa: E203
        smis.to_csv(temp_dir_chunks + str(i) + ".csv", index=False)
        nps = ftsdata[i : i + chunksize, :]  # noqa: E203
        np.save(temp_dir_chunks + str(i) + ".npy", nps)

    # could do this command line - convenient to keep within notebook for now
    # actually run predictions
    for j in range(0, len(data), chunksize):
        activate_command = "conda activate chemprop; "
        run_command = (
            "chemprop_predict --test_path "
            + temp_dir_chunks
            + str(j)
            + ".csv"
            + " --checkpoint_dir "
            + model_dir
            + " --preds_path "
            + temp_dir_chunks
            + str(j)
            + "_preds.csv"
            + " --features_path "
            + temp_dir_chunks
            + str(j)
            + ".npy --no_features_scaling --smiles_column "
            + smiles_col
            + " --ensemble_variance --gpu 0"
        )

        full_command = activate_command + run_command
        _ = subprocess.run(full_command, shell=True, capture_output=True)

    # now smush chunks back together
    df = pd.read_csv(temp_dir_chunks + "0_preds.csv")
    for i in range(chunksize, len(data), chunksize):
        new = pd.read_csv(temp_dir_chunks + str(i) + "_preds.csv")
        df = pd.concat([df, new])
    return df

# PK GNN on 37K screen

> **NOTE: This section uses the Round 0 (PK-only) GNN, which is not used in the final manuscript.** It is left for interested readers; the manuscript-reported predictions begin at the Round 1 (PK + 37K) model below.

Round 0 — pre-screen prioritization with PK-only model (`FINAL151`). **Not used in the final manuscript; left for interested readers.**

Predicts on the Broad 37K HTS library to nominate compounds for the first experimental screen against *N. gonorrhoeae*.
The PK-only GNN was trained on PK-1.7 data alone (see [2D_train_final_models_all_data.sh](2D_train_final_models_all_data.sh)).
Output `37K_chunks_with_151_model.csv` is not consumed by the downstream filter notebook and does not feed any manuscript figure.

In [ ]:
data_path = "../data/library_info/"
df = chunk_up_and_run_predictions(
    data_path="../data/library_info/",
    data_file="37Kclean.csv",
    features_file="37Kclean.npz",
    model_dir="../models/pk_screen_models_11152021/FINAL151/",
)
df.to_csv(
    "../out/predictions_from_models/pk_model/37k_screen/37K_chunks_with_151_model.csv",
    index=False,
)

# PK+37K GNN on 800K

Round 1 — first ML-based prioritization on Broad 800K (PK+37K model, `FINALbayHO04052022`).

Predicts on the Broad ~800K library using the GNN retrained on PK + 37K hits.

Output `broad800K_melis_predictions_04_25_2022.csv` is loaded by [4A_5A_filter_predictions_to_prioritize_compounds_for_validation.ipynb](4A_5A_filter_predictions_to_prioritize_compounds_for_validation.ipynb) under the *"Sets 1, 2, and 3"* exploratory section, which builds the easy / medium / hard validation tiers (`hit > 0.9` / `> 0.75` / `> 0.4` with progressively stricter Tanimoto cutoffs).

In [ ]:
data_path = "../data/library_info/"
df = chunk_up_and_run_predictions(
    data_path="../data/library_info/",
    data_file="broad800k.csv",
    features_file="broad800k.npz",
    model_dir="../models/pk_37k_screen_models_03192022/FINALbayHO04052022/",
    smiles_col="smiles",
)
df.to_csv(
    "../out/predictions_from_models/pk_37k_model/800k/broad800K_melis_predictions_04_25_2022.csv",
    index=False,
)

# PK+37K GNN on 5M 'easy-to-order' set

Round 1 — scaled to 5M "easy-to-order" library (PK+37K model, `FINALbayHO04052022`). Drives Figure 4A.

Same Round-1 model, applied to the broader ~5M commercially available compound set (Enamine REAL Diversity + neighbors, prepped in [Methods_prep_data_for_ml.ipynb](Methods_prep_data_for_ml.ipynb)).

Output `extended_screen_set_melis_predictions_05_01_2022.csv` is loaded by [4A_5A_filter_predictions_to_prioritize_compounds_for_validation.ipynb](4A_5A_filter_predictions_to_prioritize_compounds_for_validation.ipynb) in the section labeled *"Figure 4A: Round 1 model (PK+37K model) on 5M commercially available dataset"*, where it is filtered (Tanimoto < 0.5 to abx + train, hit > 0.9, PAINS/Brenk, logP < 3) and clustered to nominate the first round of validation compounds.

In [ ]:
data_path = "../data/library_info/"
df = chunk_up_and_run_predictions(
    data_path="../data/library_info/",
    data_file="cleaned_full_all_dbs_04_19_2022.csv",
    features_file="cleaned_full_all_dbs_04_19_2022.npz",
    model_dir="../models/pk_37k_screen_models_03192022/FINALbayHO04052022/",
)
df.to_csv(
    "../out/predictions_from_models/pk_37k_model/5m/extended_screen_set_melis_predictions_05_01_2022.csv",
    index=False,
)

# PK+37K+1st round screen GNN on 800K

Round 2 — model retrained with first-round validation hits, applied to Broad 800K (PK+37K+1st-round model, `FINALbayHO11152022`). Drives Figure 5A.

After experimentally validating the Round-1 picks, those hits are folded back into the training set (`data/data_prep_for_ml/data_prep_for_ml_pk_37k_first_round_val_screen/FULL_10_26_2022.csv`) and a new GNN is trained (see [2D_train_final_models_all_data.sh](2D_train_final_models_all_data.sh)).

In [ ]:
data_path = "../data/library_info/"
df = chunk_up_and_run_predictions(
    data_path="../data/library_info/",
    data_file="broad800k.csv",
    features_file="broad800k.npz",
    model_dir="../models/pk_37k_first_round_val_screen_models_10262022/FINALbayHO11152022/",
    smiles_col="smiles",
)
df.to_csv(
    "../out/predictions_from_models/pk_37k_1round_model/800k/broad800K_melis_predictions_with_FINALbayHO11152022_11_16_2022.csv",  # noqa: E501
    index=False,
)

# PK+37K+1st round screen GNN on 5M 'easy-to-order' set

Round 2 — scaled to 5M "easy-to-order" library (PK+37K+1st-round model, `FINALbayHO11152022`).

Final round of in silico prioritization: the retrained Round-2 GNN applied to the same ~5M commercial set used in Round 1, enabling head-to-head comparison.

In [ ]:
data_path = "../data/library_info/"
df = chunk_up_and_run_predictions(
    data_path="../data/library_info/",
    data_file="cleaned_full_all_dbs_04_19_2022.csv",
    features_file="cleaned_full_all_dbs_04_19_2022.npz",
    model_dir="../models/pk_37k_first_round_val_screen_models_10262022/FINALbayHO11152022/",
)
df.to_csv(
    "../out/predictions_from_models/pk_37k_1round_model/5m/extended_screen_set_with_FINALbayHO11152022_melis_predictions_11_22_2022.csv",  # noqa: E501
    index=False,
)